# Are hard-to-recognise actions harder to answer questions about?

Every QuAnTS clip shows a sequence of **four** actions drawn from a vocabulary of 19. In the human
study (paper §4.3) participants watched the animated SMPL-H clip and answered the QuAnTS questions;
in a separate identification sub-study they were additionally asked to *name* each of the four
actions (paper Fig. 9), which gives us a per-action **recognition F1** — a direct measure of how
visually hard an action is to spot.

This notebook connects the two:

1. compute the per-action recognition F1 from the identification sub-study,
2. for every action and every task split (binary / multi / open), compute the **performance delta**
   between the responses whose question *turns on* that action and the responses that do not,
3. plot the deltas and correlate them with recognition difficulty.

An action counts as involved in a response when it is named in the question text, in one of the
answer options, or in the ground-truth answer. That is deliberately narrower than "the action is
somewhere in the clip": all four actions are on screen either way, but only the named one has to be
told apart for the answer to come out right.

Metrics are the exact ones used for the paper tables (`analysis/compute-metrics.ipynb`): positive-class
F1 for binary, 3-class macro F1 for multi, and the rescaled LLM-judge score for open.

In [ ]:
import ast
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torchmetrics
from scipy import stats
from torch import Tensor, from_numpy
from torchmetrics import MetricCollection

# Work no matter whether the notebook is executed from the repo root or from analysis/.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".git").exists())
DATA = ROOT / "analysis" / "data"
STUDY = ROOT / "analysis" / "human-study"
PLOTS = ROOT / "analysis" / "plots"
PLOTS.mkdir(exist_ok=True)

sns.set_style("whitegrid")
sns.set_context("talk", font_scale=1.15)  # readable once the figures are scaled down in the paper
PALETTE = sns.color_palette("colorblind")
plt.rcParams["pdf.fonttype"] = 42  # TrueType instead of Type 3, required by most camera-ready checks
plt.rcParams["ps.fonttype"] = 42

RNG = np.random.default_rng(0)
N_BOOT = 2_000

ROOT

In [ ]:
def action_names() -> list[str]:
    """The 19 action classes, read straight out of the `ActionNames` enum in quants/data.py.

    Parsed rather than imported so that this notebook does not pull in the training dependencies
    (lightning, datasets, ...) of that module.
    """
    tree = ast.parse((ROOT / "quants" / "data.py").read_text())
    enum = next(
        node
        for node in tree.body
        if isinstance(node, ast.ClassDef) and node.name == "ActionNames"
    )
    return [n.value.value for n in enum.body if isinstance(n, ast.Assign)]


ACTIONS = action_names()
assert len(ACTIONS) == 19, ACTIONS
ACTIONS

## Human responses and the ground-truth action sequences

`all_joined_judged.h5` holds every prediction of every experiment together with the dataset labels; we
keep only the human study (`4.3` / `Human` / `organic`) on the test split.

In [ ]:
df = pd.read_hdf(DATA / "all_joined_judged.h5", key="data")
test = df[df["sample_id"] >= 27000]  # the human study only covers the test split

# sample_id -> the four ground-truth actions of that clip, in order
clip_actions = (
    test.drop_duplicates("sample_id")
    .set_index("sample_id")["action_sequence"]
    .map(lambda seq: list(seq["action"]))
)

human = test[
    (test["setting"] == "4.3") & (test["model"] == "Human") & (test["mode"] == "organic")
].copy()


def option_text(options) -> str:
    """The answer options of a question, flattened into one string ("" when there are none)."""
    if options is None or (isinstance(options, float) and pd.isna(options)):
        return ""
    if isinstance(options, dict):
        return " ".join(map(str, options.values()))
    if isinstance(options, (list, tuple, np.ndarray)):
        return " ".join(map(str, options))
    return str(options)


# Everything an action name could show up in. Lowercased, because the action names are matched
# lowercased too -- "T-posing" would otherwise never hit.
human["haystack"] = (
    human["question"].astype(str)
    + " "
    + human["options"].map(option_text)
    + " "
    + human["answer_text"].astype(str)
).str.lower()

del df, test  # the full frame is ~450 MB

print(f"{len(clip_actions)} test clips, {len(human)} human responses")
human["answer_type"].value_counts()

In [ ]:
# Every clip really has four actions, and every action is named by at least one response
assert clip_actions.map(len).eq(4).all()
named = pd.Series(
    {a: int(human["haystack"].str.contains(a.lower(), regex=False).sum()) for a in ACTIONS}
)
assert named.gt(0).all(), named[named == 0]

human[["sample_id", "question_id", "answer_type", "haystack"]].head()

## Metrics

Copied from `analysis/compute-metrics.ipynb` so that the numbers here are directly comparable to the
paper. Note that `F1Score(task="binary", average="macro")` is *not* macro-averaged — torchmetrics
ignores `average` for binary tasks, so this is the positive-class F1. Unparsable predictions are
counted as wrong: they fall below the 0.5 threshold for binary, and are explicitly forced onto a wrong
class for multi.

In [ ]:
def unwrap(res: dict[str, Tensor]) -> dict[str, float]:
    return {k: v.item() for k, v in res.items()}


def compute_binary_metrics(df: pd.DataFrame) -> dict[str, float]:
    metrics = MetricCollection(
        {
            "Accuracy": torchmetrics.Accuracy(task="binary"),
            "Precision": torchmetrics.Precision(task="binary"),
            "Recall": torchmetrics.Recall(task="binary"),
            "F1": torchmetrics.F1Score(task="binary", average="macro"),
        }
    )

    return unwrap(
        metrics(
            from_numpy(df["prediction_parsed_binary"].to_numpy(na_value=-1)),
            from_numpy(df["answer"].to_numpy()),
        )
    )


def compute_multi_metrics(df: pd.DataFrame) -> dict[str, float]:
    metrics = MetricCollection(
        {
            "Accuracy": torchmetrics.Accuracy(task="multiclass", num_classes=3),
            "Precision": torchmetrics.Precision(
                task="multiclass", average="macro", num_classes=3
            ),
            "Recall": torchmetrics.Recall(task="multiclass", average="macro", num_classes=3),
            "F1": torchmetrics.F1Score(task="multiclass", num_classes=3, average="macro"),
        }
    )

    ground_truth = df["answer"].to_numpy().astype(np.int64)

    predictions = (
        df["prediction_parsed_multi"]
        .map({"A": 0, "B": 1, "C": 2, "nan": pd.NA})
        .to_numpy(na_value=-1)
        .astype(np.int32)
    )
    # Make sure we put a wrong answer in case of NaN
    predictions[predictions == -1] = (ground_truth[predictions == -1] + 1) % 3

    return unwrap(metrics(from_numpy(predictions), from_numpy(ground_truth)))


def compute_open_llm_judge(df: pd.DataFrame) -> float:
    """The LLM-judge score, rescaled from [1, 3] to [0, 1] as in the paper."""
    return (df["llm_judge_score"].fillna(1).mean() - 1) / 2

In [ ]:
binary = human[human["answer_type"] == "binary"].copy()
binary["prediction_parsed_binary"] = binary["prediction_parsed_binary"].fillna(np.nan)
binary = binary.astype({"answer": int, "prediction_parsed_binary": float})

multi = human[human["answer_type"] == "multi"].copy()
multi["prediction_parsed_multi"] = multi["prediction_parsed_multi"].fillna(np.nan)

open_ = human[human["answer_type"] == "open"].copy()

SPLITS = {"binary": binary, "multi": multi, "open": open_}
{name: len(split) for name, split in SPLITS.items()}

### Check: do we match the published human reference scores?

The whole-group numbers must match the `4.3 / Human / organic` row of the result files that back
Tables 2 and 3 of the paper.

In [ ]:
published = {
    name: pd.read_csv(DATA / f"all_results_{name}.csv")
    .set_index(["setting", "model", "mode"])
    .loc[("4.3", "Human", "organic")]
    for name in SPLITS
}

reference_scores = {
    "binary": compute_binary_metrics(binary)["F1"],
    "multi": compute_multi_metrics(multi)["F1"],
    "open": compute_open_llm_judge(open_),
}

for name, value in reference_scores.items():
    reference = published[name]["LLMJudge" if name == "open" else "F1"]
    print(f"{name:>6}: {value:.6f} (published {reference:.6f})")
    assert np.isclose(value, reference, atol=1e-6), name

### Fast re-implementations for bootstrapping

Bootstrapping means ~100 k metric evaluations, which is far too slow through torchmetrics. These NumPy
twins are asserted to agree with it exactly.

In [ ]:
def f1_binary(predictions: np.ndarray, targets: np.ndarray) -> float:
    """Positive-class F1, matching `F1Score(task="binary")`."""
    positive = predictions == 1
    tp = np.count_nonzero(positive & (targets == 1))
    fp = np.count_nonzero(positive & (targets == 0))
    fn = np.count_nonzero(~positive & (targets == 1))
    denominator = 2 * tp + fp + fn
    return 2 * tp / denominator if denominator else 0.0


def f1_macro_multi(predictions: np.ndarray, targets: np.ndarray) -> float:
    """Macro-averaged F1 over the three answer options."""
    scores = []
    for c in range(3):
        predicted = predictions == c
        actual = targets == c
        tp = np.count_nonzero(predicted & actual)
        fp = np.count_nonzero(predicted & ~actual)
        fn = np.count_nonzero(~predicted & actual)
        denominator = 2 * tp + fp + fn
        scores.append(2 * tp / denominator if denominator else 0.0)
    return float(np.mean(scores))


# Pre-extract the per-response arrays each metric needs, so that a bootstrap resample is just indexing
binary_predictions = (binary["prediction_parsed_binary"].to_numpy(na_value=-1) == 1).astype(np.int64)
binary_targets = binary["answer"].to_numpy().astype(np.int64)

multi_targets = multi["answer"].to_numpy().astype(np.int64)
multi_predictions = (
    multi["prediction_parsed_multi"]
    .map({"A": 0, "B": 1, "C": 2, "nan": pd.NA})
    .to_numpy(na_value=-1)
    .astype(np.int64)
)
multi_predictions[multi_predictions == -1] = (multi_targets[multi_predictions == -1] + 1) % 3

open_scores = open_["llm_judge_score"].fillna(1).to_numpy(dtype=float)

METRICS = {
    "binary": lambda idx: f1_binary(binary_predictions[idx], binary_targets[idx]),
    "multi": lambda idx: f1_macro_multi(multi_predictions[idx], multi_targets[idx]),
    "open": lambda idx: (open_scores[idx].mean() - 1) / 2,
}
METRIC_LABELS = {"binary": "F1", "multi": "Macro F1", "open": "LLM judge"}

for name, metric in METRICS.items():
    everything = np.arange(len(SPLITS[name]))
    assert np.isclose(metric(everything), reference_scores[name], atol=1e-9), name
print("NumPy implementations agree with torchmetrics")

## Recognition difficulty (the scores behind Fig. 9)

In the identification sub-study each of the 440 open-split clips was shown to one participant who named
all four actions, choosing `<not recognized>` when they could not tell. Scoring those names against the
ground-truth sequence gives a per-action F1: an action is a false negative when it was present but named
something else (or not recognised), and a false positive when it was named for a slot holding a
different action.

In [ ]:
identification = pd.read_csv(STUDY / "open" / "identification_results_open.csv").merge(
    pd.read_csv(STUDY / "open" / "study_data_open.csv")[["study_entry_id", "sample_id"]],
    left_on="study_entry_ids",
    right_on="study_entry_id",
    validate="many_to_one",
)
identification["truth"] = [
    clip_actions[sample][index]
    for sample, index in zip(identification["sample_id"], identification["action_index"])
]

not_recognised = (identification["answer"] == "<not recognized>").mean()
print(
    f"{len(identification)} slots over {identification['sample_id'].nunique()} clips, "
    f"{not_recognised:.2%} not recognised, "
    f"{(identification['answer'] == identification['truth']).mean():.2%} named correctly"
)
identification.head()

In [ ]:
def recognition_scores(action: str) -> dict[str, float]:
    named = identification["answer"] == action
    present = identification["truth"] == action

    tp = np.count_nonzero(named & present)
    fp = np.count_nonzero(named & ~present)
    fn = np.count_nonzero(~named & present)
    denominator = 2 * tp + fp + fn

    return {
        "action": action,
        "support": int(present.sum()),
        "recognition_f1": 2 * tp / denominator if denominator else 0.0,
    }


recognition = pd.DataFrame([recognition_scores(action) for action in ACTIONS]).set_index(
    "action"
)

# Easiest first -- this order is reused by every figure below
ACTION_ORDER = recognition["recognition_f1"].sort_values(ascending=False).index.tolist()
recognition.loc[ACTION_ORDER]

## Performance delta per action

For each action and each split: the metric over the responses that name the action (question, options
or answer) minus the metric over the responses that do not. The 95 % interval comes from 2 000
bootstrap resamples drawn independently within each of the two groups, so it reflects the group
sizes, which are small — roughly 40 responses per action for binary and open, 120 for multi, where
every question offers three options.

In [ ]:
def delta_with_ci(
    metric, inside: np.ndarray, outside: np.ndarray
) -> tuple[float, float, float]:
    """Metric difference between the two groups, with a percentile bootstrap interval."""
    delta = metric(inside) - metric(outside)

    resampled = np.array(
        [
            metric(RNG.choice(inside, size=inside.size, replace=True))
            - metric(RNG.choice(outside, size=outside.size, replace=True))
            for _ in range(N_BOOT)
        ]
    )
    low, high = np.percentile(resampled, [2.5, 97.5])
    return delta, low, high


records = []
for name, split in SPLITS.items():
    metric = METRICS[name]
    for action in ACTIONS:
        present = split["haystack"].str.contains(action.lower(), regex=False).to_numpy()
        inside = np.flatnonzero(present)
        outside = np.flatnonzero(~present)
        assert inside.size + outside.size == len(split)

        delta, low, high = delta_with_ci(metric, inside, outside)
        records.append(
            {
                "action": action,
                "split": name,
                "n_in": inside.size,
                "n_out": outside.size,
                "metric_in": metric(inside),
                "metric_out": metric(outside),
                "delta": delta,
                "ci_low": low,
                "ci_high": high,
            }
        )

deltas = pd.DataFrame(records).join(recognition, on="action")
deltas.head()

In [ ]:
deltas.to_csv(DATA / "action_difficulty_deltas.csv", index=False)

(
    deltas.assign(**{c: deltas[c] * 100 for c in ["delta", "ci_low", "ci_high"]})
    .pivot(index="action", columns="split", values="delta")
    .loc[ACTION_ORDER]
    .round(2)
)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 9), sharey=True)
positions = np.arange(len(ACTION_ORDER))

for ax, (name, colour) in zip(axes, zip(SPLITS, PALETTE)):
    subset = deltas[deltas["split"] == name].set_index("action").loc[ACTION_ORDER]
    values = subset["delta"].to_numpy() * 100
    errors = np.abs(
        np.stack([subset["ci_low"], subset["ci_high"]]) * 100 - values
    )

    ax.barh(positions, values, color=colour, height=0.75, zorder=2)
    ax.errorbar(
        values,
        positions,
        xerr=errors,
        fmt="none",
        ecolor="0.25",
        elinewidth=1.2,
        capsize=3,
        zorder=3,
    )
    ax.axvline(0, color="0.2", linewidth=1.2, zorder=1)
    ax.set_title(f"{name.capitalize()} ({METRIC_LABELS[name]})")
    ax.grid(True, axis="x", linestyle="-", alpha=0.5)
    ax.grid(False, axis="y")
    sns.despine(ax=ax, top=False, right=False)

    # In-group size in its own gutter on the right, where it cannot collide with a bar
    gutter = ax.secondary_yaxis("right")
    gutter.set_yticks(
        positions, [str(n) for n in subset["n_in"]], fontsize="small", color="0.35"
    )
    gutter.tick_params(axis="y", length=0, pad=2)
    ax.annotate(
        "n",
        xy=(1, 1),
        xycoords="axes fraction",
        xytext=(6, 6),
        textcoords="offset points",
        ha="left",
        va="baseline",
        fontsize="small",
        color="0.35",
    )

# The one action name long enough to stretch the figure, shortened as in the paper
ABBREVIATED = {"picking something up with both hands": "picking [...] both hands"}

axes[0].set_yticks(
    positions,
    [
        f"{ABBREVIATED.get(action, action)} ({recognition.loc[action, 'recognition_f1']:.0%})"
        for action in ACTION_ORDER
    ],
)
axes[0].invert_yaxis()  # easiest to recognise at the top
axes[0].set_ylim(len(ACTION_ORDER) - 0.5, -0.5)
axes[1].set_xlabel(
    "Change in human performance when the action is named in the question or answer "
    "[percentage points]"
)

# Heading for the action column, typeset like the panel titles next to it
axes[0].annotate(
    "Action (Recognition F1)",
    xy=(0, 1),
    xycoords="axes fraction",
    xytext=(-10, 6),
    textcoords="offset points",
    ha="right",
    va="baseline",
    fontsize=axes[0].title.get_fontsize(),
)

fig.tight_layout()
fig.savefig(PLOTS / "action_deltas.pdf", bbox_inches="tight", pad_inches=0)
plt.show()

## Does recognition difficulty explain the delta?

Each panel correlates the 19 per-action recognition F1 values against the 19 per-action deltas of that
split, so `n = 19` — the actions are the unit here, not the individual responses. Pearson's r and
Spearman's rho are both reported; rho is the more trustworthy of the two, because a single action
(`catching a ball`, 31 % recognition) sits far from the rest on the x axis and pulls the linear fit.

The p values are those of the individual coefficients, uncorrected. Multiplicity enters only when the
panels are read jointly as one claim ("the effect shows up somewhere"), which would put the Bonferroni
threshold at 0.05 / 3 = 0.017 — open clears it on both coefficients, binary and multi on neither. Note
also that the 19 deltas within a split are not independent of each other — they are estimated from
overlapping response groups — so the p values are somewhat optimistic to begin with.

In [ ]:
def format_p(p: float) -> str:
    """The p value of this one coefficient, uncorrected -- see the note above on multiplicity."""
    return "p < 0.001" if p < 0.001 else f"p = {p:.3f}"


fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharex=True)

for ax, (name, colour) in zip(axes, zip(SPLITS, PALETTE)):
    subset = deltas[deltas["split"] == name]
    x = subset["recognition_f1"].to_numpy() * 100
    y = subset["delta"].to_numpy() * 100

    sns.regplot(
        x=x,
        y=y,
        ax=ax,
        color=colour,
        scatter_kws=dict(s=60, alpha=0.85, zorder=3),
        line_kws=dict(linewidth=2, zorder=2),
    )
    ax.axhline(0, color="0.2", linewidth=1.2, zorder=1)

    pearson = stats.pearsonr(x, y)
    spearman = stats.spearmanr(x, y)
    ax.set_title(f"{name.capitalize()} ({METRIC_LABELS[name]})")
    ax.text(
        0.04,
        0.04,
        f"r = {pearson.statistic:.2f} ({format_p(pearson.pvalue)})\n"
        f"\u03c1 = {spearman.statistic:.2f} ({format_p(spearman.pvalue)})",
        transform=ax.transAxes,
        va="bottom",
        ha="left",
        fontsize="small",
        bbox=dict(boxstyle="round", facecolor="white", edgecolor="0.8", alpha=0.9),
    )
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.grid(True, linestyle="-", alpha=0.5)
    sns.despine(ax=ax, top=False, right=False)

axes[0].set_ylabel("\u0394 performance [percentage points]")
axes[1].set_xlabel("Recognition F1 of the action")
for ax in axes:
    ax.set_xticks(np.arange(20, 101, 20), [f"{x}%" for x in range(20, 101, 20)])

fig.tight_layout()
fig.savefig(PLOTS / "action_recognition_vs_delta.pdf", bbox_inches="tight", pad_inches=0)
plt.show()

In [ ]:
# How many of the 57 deltas are individually distinguishable from zero?
significant = deltas[(deltas["ci_low"] > 0) | (deltas["ci_high"] < 0)]
print(f"{len(significant)} of {len(deltas)} deltas have a 95% interval excluding zero")
significant[["split", "action", "n_in", "n_out", "delta", "ci_low", "ci_high", "recognition_f1"]]